# Delta Live Tables Pipeline: Retail Analytics
This notebook defines a DLT pipeline for retail analytics using the TPC-H dataset.


In [0]:
# Configure the pipeline
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *


## Bronze Layer
The bronze layer ingests raw data from source systems with minimal transformation.


In [0]:

import dlt
from pyspark.sql.functions import current_timestamp, col
#create streaming view for customer
@dlt.view(
    name="bronze_customers_vw",
    comment="Raw customer data from ecommerce.rw.customer_rw"
)
def bronze_customers():
    return (spark.readStream.table("ecommerce.rw.customer_rw")
            .withColumn("ingestion_ts", current_timestamp())) 


#creating streaming table for customer scd 2
dlt.create_streaming_table(
    name="silver_customer_scd_2",
    comment="Customer dimension SCD Type 2"
)

# ---- Apply SCD2 changes ----
dlt.apply_changes(
    target="silver_customer_scd_2",          
    source="live.bronze_customers_vw",               
    keys=["c_custkey"],
    sequence_by=col("ingestion_ts"), 
    stored_as_scd_type=2,
    track_history_column_list=["c_custkey", "ingestion_ts"]
)


In [0]:
# @dlt.table(
#     name="bronze_customers",
#     comment="Raw customer data from TPC-H dataset"
# )
# def bronze_customers():
#     df = spark.read.table("ecommerce.rw.customer_rw").withColumn("injestion_date", date_format(current_timestamp(),"yyyy-MM-dd HH:mm:ss") )
#     return df



In [0]:

# dlt.create_table(
#     name="silver_customer_scd_2",
  
# )

In [0]:
# dlt.apply_changes(
#     source="Live.bronze_customers",
#     target="ecommerce.dbo.silver_customer_scd_2",
#     keys = ["c_custkey"],
#     sequence_by = col("injestion_date"),
#     stored_as_scd_type=2,
#     track_history_column_list=["c_custkey", "injestion_date"]
# )

In [0]:
@dlt.table(
    name="bronze_orders",
    comment="Raw orders data from TPC-H dataset"
)
def bronze_orders():
    df = spark.readStream.table("ecommerce.rw.orders_rw")
    return df

#create a streaming table for the autoloader order 
@dlt.table(
    comment = "Order Autoloader ",
    table_properties = {
        "quality" : "bronze"
    },
    name = "bronze_orders_autoloader"
)
def func():
    df = spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "csv")\
        .option("cloudFiles.schemaHints","o_orderkey long, o_custkey long,o_orderstatus string,o_totalprice decimal(18,2),o_orderdate date,o_orderpriority string,o_clerk string,o_shippriority integer,o_comment string")\
            .option("cloudFiles.schemaLocation","/Volumes/ecommerce/rw/landing/autoloaders/schemas/1")\
            .option("cloudfiles.format", "csv")\
            .option("pathGlobFilter", "*.csv")\
            .option("cloudfiles.schemaEvolutionMode","none")\
            .load("/Volumes/ecommerce/rw/landing/files/")
    return df

#streaming table to union both the autoloader and the streaming table using append flow
dlt.create_streaming_table("bronze_orders_union")
#Append flow
@dlt.append_flow(
    target = "bronze_orders_union",

)
def order_delta_append():
    df = spark.readStream.table("Live.bronze_orders")
    return df
#Append flow
@dlt.append_flow(
    target = "bronze_orders_union",

)
def order_autoloader_append():
    df = spark.readStream.table("Live.bronze_orders_autoloader")
    return df

In [0]:

#create bronze table for line item
@dlt.table(
    name="bronze_lineitem",
    comment="Raw lineitem data from lineitem dataset"
)
def bronze_lineitem():
    df = spark.readStream.table("ecommerce.rw.lineitem_rw")
    return df


## Silver Layer
The silver layer cleans, validates, and standardizes data from the bronze layer.


In [0]:
#creating materialized view for silver customer with expectations 
@dlt.table(
    name="silver_customers_active_records",
    comment="Cleaned and validated customer data"
)
@dlt.expect_all({
    "valid_customer_key": "c_custkey IS NOT NULL",
    "valid_market_segment": "c_mktsegment IS NOT NULL"
})
def silver_customers():
    return dlt.read("silver_customer_scd_2").filter(col("__END_AT").isNull()).select(
        col("c_custkey"),
        col("c_name"),
        col("c_address"),
        col("c_nationkey"),
        col("c_phone"),
        col("c_acctbal"),
        col("c_mktsegment"),
        col("c_comment")
    )

In [0]:
#creating materialized view for silver order with expectations 
@dlt.table(
    name="silver_orders",
    comment="Cleaned and validated orders data"
)
@dlt.expect_all({
    "valid_order_key": "o_orderkey IS NOT NULL",
    "valid_customer_key": "o_custkey IS NOT NULL",
    "valid_order_date": "o_orderdate IS NOT NULL",
    "valid_total_price": "o_totalprice > 0"
})
def silver_orders():
    return dlt.read("bronze_orders_union").select(
        col("o_orderkey"),
        col("o_custkey"),
        col("o_orderstatus"),
        col("o_totalprice"),
        col("o_orderdate"),
        col("o_orderpriority"),
        col("o_clerk"),
        col("o_shippriority"),
        col("o_comment")
    )

In [0]:
#creating materialized view for silver_lineitem with expectations 
@dlt.table(
    name="silver_lineitem",
    comment="Cleaned and validated lineitem data"
)
@dlt.expect_all({
    "valid_order_key": "l_orderkey IS NOT NULL",
    "valid_quantity": "l_quantity > 0",
    "valid_price": "l_extendedprice > 0"
})
def silver_lineitem():
    return dlt.read("bronze_lineitem").select(
        col("l_orderkey"),
        col("l_partkey"),
        col("l_suppkey"),
        col("l_linenumber"),
        col("l_quantity"),
        col("l_extendedprice"),
        col("l_discount"),
        col("l_tax"),
        col("l_returnflag"),
        col("l_linestatus"),
        col("l_shipdate"),
        col("l_commitdate"),
        col("l_receiptdate"),
        col("l_shipinstruct"),
        col("l_shipmode"),
        col("l_comment")
    ).withColumn("revenue", col("l_extendedprice") * (1 - col("l_discount")))

## Gold Layer
The gold layer creates business-ready tables for analytics and reporting.


In [0]:
#creating gold customer orders summary
@dlt.table(
    name="gold_customer_orders",
    comment="Customer orders summary for analytics"
)
def gold_customer_orders():
    return dlt.read("silver_orders").alias("o") \
        .join(
            dlt.read("silver_customers_active_records").alias("c"),
            col("o.o_custkey") == col("c.c_custkey") ,
            "inner"
        ) \
        .select(
            col("c.c_custkey"),
            col("c.c_name"),
            col("c.c_mktsegment"),
            col("o.o_orderkey"),
            col("o.o_orderdate"),
            col("o.o_totalprice"),
            year(col("o.o_orderdate")).alias("order_year"),
            month(col("o.o_orderdate")).alias("order_month")
        )

In [0]:


# creating gold aggregates by market segment
@dlt.table(
    name="gold_monthly_sales",
    comment="Monthly sales aggregated by market segment"
)
def gold_monthly_sales():
    return dlt.read("gold_customer_orders") \
        .groupBy("order_year", "order_month", "c_mktsegment") \
        .agg(
            sum("o_totalprice").alias("total_sales"),
            count("o_orderkey").alias("order_count"),
            countDistinct("c_custkey").alias("customer_count")
        ) \
        .orderBy("order_year", "order_month", "c_mktsegment")




In [0]:

#creating performance analysis table 
@dlt.table(
    name="gold_product_performance",
    comment="Product performance analysis"
)
def gold_product_performance():
    return dlt.read("silver_lineitem").alias("li") \
        .join(
            dlt.read("silver_orders").alias("o"),
            col("li.l_orderkey") == col("o.o_orderkey"),
            "inner"
        ) \
        .groupBy("li.l_partkey") \
        .agg(
            sum("li.revenue").alias("total_revenue"),
            sum("li.l_quantity").alias("total_quantity"),
            count("li.l_orderkey").alias("order_count"),
            avg("li.revenue").alias("avg_revenue_per_order")
        ) \
        .orderBy(col("total_revenue").desc())